# Práctica 4 · Procesamiento del vídeo de test

Prototipo que procesa el vídeo de test `C0142.MP4` fotograma a fotograma y:

1. **Detecta y sigue** personas y vehículos (YOLO11n + `model.track`, BoT-SORT) y **cuenta** los objetos únicos de cada clase.
2. **Detecta la matrícula** de cada vehículo con el modelo propio `matriculas_yolo11n.pt` (entrenado en `P4_colab_entrenamiento_y_ocr.ipynb`).
3. **Lee la matrícula** con **SmolVLM** (P4b) y se queda, para cada vehículo, con la lectura más votada a lo largo de los fotogramas.
4. Guarda el **vídeo de resultados** (`resultado.mp4`) y el **CSV** (`resultados.csv`, separador `;`).
5. Extras: **flujo por direcciones** y **vídeo anonimizado** (`resultado_anonimizado.mp4`).

Se parte de la celda *Seguimiento* del cuaderno `VC_P4.ipynb` de la asignatura y del ejemplo de SmolVLM de `VC_P4b.ipynb`. En el código se marca qué viene de la **PLANTILLA del profesor**, qué se ha **MODIFICADO** y qué es **NUEVO**.

Se ejecutó en **Google Colab con GPU NVIDIA L4**: SmolVLM en un portátil sin GPU tardaría varios segundos por matrícula. Por eso no hay ventana de visualización (`cv2.imshow` no funciona en Colab) y los archivos se leen y escriben en la carpeta `VC_P4` de Google Drive.

## Preparación del entorno

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics lap
!nvidia-smi --query-gpu=name --format=csv

Mounted at /content/drive
name
NVIDIA L4


## Carga de SmolVLM

Misma configuración que en la comparativa OCR de P4b: GPU, `bfloat16`, `_attn_implementation="eager"` (en Colab no está instalado `flash_attention_2`), *prompt* que pide solo los caracteres y el mismo post-procesado (`limpiar`).

In [ ]:
import os, re, csv, math
import cv2
import numpy as np
import torch
from collections import defaultdict, Counter
from ultralytics import YOLO                              # PLANTILLA del profesor (primera celda)
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

# NUEVO: trabajar en la carpeta de Google Drive, donde están el vídeo y el modelo
os.chdir('/content/drive/MyDrive/VC_P4')
print('Vídeo:', os.path.exists('C0142.MP4'), '| Modelo:', os.path.exists('matriculas_yolo11n.pt'))

# PLANTILLA del profesor (P4b): SmolVLM; MODIFICADO: GPU y "eager" (igual que en la comparativa OCR)
Device = "cuda"
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
model_vlm = AutoModelForImageTextToText.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                        dtype=torch.bfloat16,
                                                        _attn_implementation="eager").to(Device)
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": "Can you give me the text in the license plate of the image? "
                             "Answer only with the characters."}]}]               # MODIFICADO: pedir solo los caracteres
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

def limpiar(texto):                                       # NUEVO (OCR): mismo post-procesado que en la comparativa
    t = re.sub(r'[^A-Z0-9]', '', texto.upper())
    if len(t) > 1 and t[0] == 'E' and t[1].isdigit():     # quitar la "E" de la franja azul de la UE
        t = t[1:]
    return t

@torch.no_grad()
def leer_matricula_vlm(recorte):                          # NUEVO (OCR): misma función que en la comparativa OCR
    imagen = Image.fromarray(cv2.cvtColor(recorte, cv2.COLOR_BGR2RGB))
    inputs = processor(text=prompt, images=[imagen], return_tensors="pt").to(Device)
    generated_ids = model_vlm.generate(**inputs, max_new_tokens=16)
    nuevos = generated_ids[:, inputs['input_ids'].shape[1]:]                     # solo la respuesta, sin la pregunta
    return limpiar(processor.batch_decode(nuevos, skip_special_tokens=True)[0])

print("SmolVLM cargado")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Vídeo: True | Modelo: True


SmolVLM cargado


## Detección, seguimiento, matrículas y OCR

- **Matrículas:** el detector de matrículas se aplica sobre el **recorte de cada vehículo** (coche, moto, autobús o camión), donde la matrícula se ve más grande que en el fotograma completo. Solo en vehículos de al menos 120 × 80 px.
- **OCR:** cada vehículo se lee con SmolVLM como mucho cada 5 fotogramas y solo si la matrícula mide al menos 60 px de ancho. Se descartan lecturas de menos de 4 o más de 8 caracteres. Todas las lecturas se guardan por ID y se muestra la **más votada** (cada lectura cuenta 1, porque SmolVLM no devuelve una confianza).
- **Dirección:** se compara la primera y la última posición del centro de cada objeto; si se ha movido menos del 5 % del ancho de la imagen se considera *parado*.
- **Anonimización:** en una copia limpia de cada fotograma se pixelan todas las personas y vehículos detectados.

La salida de abajo corresponde al procesamiento del vídeo completo.

In [ ]:
model = YOLO('yolo11n.pt')
modelo_matriculas = YOLO('matriculas_yolo11n.pt')         # NUEVO (matrículas): modelo entrenado en Colab (YOLO11n + UC3M-LP)
CLASES_VEHICULO = [2, 3, 5, 7]                            # NUEVO (matrículas): car, motorcycle, bus, truck

votos = defaultdict(Counter)                              # NUEVO (OCR): por ID, todas las lecturas
ultimo_ocr = defaultdict(lambda: -100)                    # NUEVO (OCR): fotograma de la última lectura por ID

vid = cv2.VideoCapture("C0142.MP4")                       # MODIFICADO: vídeo en lugar de la webcam
total = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

# NUEVO: writer para el vídeo de resultados (mismo tamaño y fps que el original)
fps = vid.get(cv2.CAP_PROP_FPS)
ancho = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter("resultado.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

# NUEVO (anonimización): segundo vídeo
writer_anon = cv2.VideoWriter("resultado_anonimizado.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

def pixelar(imagen, x1, y1, x2, y2, bloques=10):          # NUEVO (anonimización)
    """Pixela la región (x1, y1)-(x2, y2) directamente en la imagen."""
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(imagen.shape[1], x2), min(imagen.shape[0], y2)
    if x2 - x1 < 2 or y2 - y1 < 2:
        return
    zona = imagen[y1:y2, x1:x2]
    pequena = cv2.resize(zona, (bloques, bloques), interpolation=cv2.INTER_LINEAR)
    imagen[y1:y2, x1:x2] = cv2.resize(pequena, (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)

ids_por_clase = defaultdict(set)                          # NUEVO: IDs de seguimiento por clase para el conteo
track_history = defaultdict(lambda: [])                   # PLANTILLA del profesor: posiciones por ID
clase_por_id = {}                                         # NUEVO (dirección): clase de cada ID

# NUEVO: abrir el CSV y escribir la cabecera (columnas según el README de la práctica)
archivo_csv = open("resultados.csv", "w", newline="", encoding="utf-8-sig")
csv_writer = csv.writer(archivo_csv, delimiter=";")
csv_writer.writerow(["fotograma", "tipo_objeto", "confianza", "identificador_tracking",
                     "x1", "y1", "x2", "y2",
                     "matrícula_en_su_caso", "confianza_matricula",
                     "mx1", "my1", "mx2", "my2", "texto_matricula"])

fotograma = 0                                             # NUEVO: contador de fotogramas
lecturas_ocr = 0                                          # NUEVO (OCR): número de llamadas a SmolVLM

while(True):
    ret, img = vid.read()
    if not ret:                                           # NUEVO: parar al final del vídeo
        break

    img_original = img.copy()                             # NUEVO (matrículas): copia sin dibujos para los recortes
    img_anon = img_original.copy()                        # NUEVO (anonimización): copia limpia para pixelar

    results = model.track(img, persist=True,
                          classes=[0,1,2,3,5,7],          # MODIFICADO: + moto, autobús, camión
                          verbose=False)                  # NUEVO: sin salida de texto por fotograma

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            if box.id is not None:
                track_id = str(int(box.id[0].tolist()))
            else:
                track_id = ''

            confidence = math.ceil((box.conf[0]*100))/100
            cls = int(box.cls[0])                         # MODIFICADO: eliminados los print

            pixelar(img_anon, x1, y1, x2, y2)             # NUEVO (anonimización): pixelar persona/vehículo

            if track_id != '':                            # NUEVO: añadir la ID al conjunto de su clase
                ids_por_clase[model.names[cls]].add(track_id)

                # NUEVO (dirección): guardar el centro de la caja (idea del track_history del profesor)
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                track_history[track_id].append((cx, cy))
                clase_por_id[track_id] = model.names[cls]

                # PLANTILLA del profesor (bloque if 0): dibujar la trayectoria de las últimas 30 posiciones
                puntos = np.array(track_history[track_id][-30:], dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(img, [puntos], isClosed=False, color=(230, 230, 230), thickness=3)

            # NUEVO (matrículas): valores por defecto si no se encuentra matrícula
            hay_matricula = 0
            conf_mat, mx1, my1, mx2, my2 = "", "", "", "", ""
            texto_matricula = ""                          # NUEVO (OCR)

            # NUEVO (matrículas): buscar la matrícula solo en el recorte de los vehículos
            if cls in CLASES_VEHICULO:
                recorte = img_original[y1:y2, x1:x2]
                if (x2 - x1) > 120 and (y2 - y1) > 80:            # solo vehículos suficientemente grandes
                    res_mat = modelo_matriculas(recorte, conf=0.4, imgsz=320, verbose=False)[0]
                    if len(res_mat.boxes) > 0:
                        mejor = int(res_mat.boxes.conf.argmax())        # matrícula con mayor confianza
                        px1, py1, px2, py2 = res_mat.boxes.xyxy[mejor]
                        mx1, my1 = x1 + int(px1), y1 + int(py1)          # pasar a coordenadas del fotograma completo
                        mx2, my2 = x1 + int(px2), y1 + int(py2)
                        conf_mat = math.ceil((res_mat.boxes.conf[mejor]*100))/100
                        hay_matricula = 1
                        cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 255), 2)   # recuadro amarillo

                        # NUEVO (OCR): leer la matrícula si es suficientemente grande y no se ha leído hace poco
                        if track_id != '' and (mx2 - mx1) >= 60 and fotograma - ultimo_ocr[track_id] >= 5:
                            ultimo_ocr[track_id] = fotograma
                            texto = leer_matricula_vlm(img_original[my1:my2, mx1:mx2])   # MODIFICADO: SmolVLM en lugar de EasyOCR
                            lecturas_ocr += 1
                            if 4 <= len(texto) <= 8:                                     # descartar lecturas demasiado cortas o largas
                                votos[track_id][texto] += 1                               # MODIFICADO: SmolVLM no da confianza, cada lectura cuenta 1

                        # NUEVO (OCR): se muestra la lectura más votada hasta ahora de este vehículo
                        if track_id != '' and votos[track_id]:
                            texto_matricula = votos[track_id].most_common(1)[0][0]
                            cv2.putText(img, texto_matricula, (mx1, my2 + 30),
                                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

            # MODIFICADO (OCR): ahora se rellena texto_matricula
            csv_writer.writerow([fotograma, model.names[cls], confidence, track_id,
                                 x1, y1, x2, y2,
                                 hay_matricula, conf_mat, mx1, my1, mx2, my2, texto_matricula])

            escala = int((cls / len(model.names)) * 255 * 3)   # MODIFICADO: model.names
            if escala >= 255*2:
                R, G, B = 255, 255, escala - 255*2
            elif escala >= 255:
                R, G, B = 255, escala - 255, 0
            else:
                R, G, B = escala, 0, 0

            cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
            cv2.putText(img, track_id + ' ' + model.names[cls], [x1, y1],   # MODIFICADO: model.names
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

    writer.write(img)                                     # NUEVO: escribir el fotograma en el vídeo de resultados
    writer_anon.write(img_anon)                           # NUEVO (anonimización)

    fotograma += 1                                        # NUEVO: siguiente fotograma
    if fotograma % 250 == 0:                              # NUEVO: progreso (en Colab no hay ventana de visualización)
        print(f"{fotograma}/{total} fotogramas · {lecturas_ocr} lecturas OCR")

vid.release()
writer.release()                                          # NUEVO: cerrar correctamente el archivo de vídeo
writer_anon.release()                                     # NUEVO (anonimización)
archivo_csv.close()                                       # NUEVO: cerrar correctamente el CSV

# NUEVO: mostrar el resultado del conteo
print("\nNúmero de objetos por clase:")
for clase, ids in ids_por_clase.items():
    print(clase, ":", len(ids))

# NUEVO (dirección): comparar la primera y la última posición y contar por clase
umbral = 0.05 * ancho                                     # menos del 5 % del ancho de la imagen = parado
direcciones = defaultdict(lambda: defaultdict(int))
for tid, posiciones in track_history.items():
    (xa, ya), (xb, yb) = posiciones[0], posiciones[-1]
    dx, dy = xb - xa, yb - ya
    if max(abs(dx), abs(dy)) < umbral:
        direccion = "parado"
    elif abs(dx) >= abs(dy):
        direccion = "derecha" if dx > 0 else "izquierda"
    else:
        direccion = "abajo" if dy > 0 else "arriba"
    direcciones[clase_por_id[tid]][direccion] += 1

print("\nFlujo por direcciones:")
for clase, conteo in direcciones.items():
    print(clase, ":", dict(conteo))

# NUEVO (OCR): matrícula leída por vehículo
print("\nMatrículas leídas:")
for tid, c in votos.items():
    if c:                                                 # saltar vehículos sin lectura
        print("ID", tid, "(" + clase_por_id.get(tid, "?") + "):", c.most_common(1)[0][0])

250/2832 fotogramas · 1 lecturas OCR
500/2832 fotogramas · 2 lecturas OCR
750/2832 fotogramas · 4 lecturas OCR
1000/2832 fotogramas · 14 lecturas OCR
1250/2832 fotogramas · 21 lecturas OCR
1500/2832 fotogramas · 42 lecturas OCR
1750/2832 fotogramas · 63 lecturas OCR
2000/2832 fotogramas · 77 lecturas OCR
2250/2832 fotogramas · 87 lecturas OCR
2500/2832 fotogramas · 95 lecturas OCR
2750/2832 fotogramas · 105 lecturas OCR

Número de objetos por clase:
bus : 2
car : 43
truck : 2
person : 16
motorcycle : 2
bicycle : 1

Flujo por direcciones:
car : {'izquierda': 25, 'parado': 12, 'derecha': 5}
person : {'derecha': 10, 'parado': 1, 'izquierda': 3}
motorcycle : {'derecha': 1, 'izquierda': 1}
bicycle : {'izquierda': 1}

Matrículas leídas:
ID 1 (car): 322555
ID 8 (car): 1234
ID 10 (car): 515957
ID 9 (car): 3713LKC
ID 11 (car): KFST
ID 13 (car): 5555
ID 12 (car): 0848MCG
ID 15 (car): 6713HG3
ID 18 (car): 2314FGN
ID 21 (car): 5726
ID 24 (car): 2537
ID 23 (car): 0053JFW
ID 26 (car): 0285FZ
ID 32 (